# Sequential and Hierarchical Workflows

Not every problem fits a single group chat. **Sequential** workflows run agent A, stop, then agent B on a **new task** or continued thread. **Hierarchical** workflows add a **manager** that delegates to workers — similar to **LangGraph/11** supervisor graphs.

This notebook builds **Notes API pipelines**, previews **`MagenticOneGroupChat`**, and compares **pipeline vs conversation** orchestration.

**Prerequisites:** **09. GroupChatManager and Speaker Selection**, **10. Custom Agent Roles**, **13. Termination Conditions**.

**Cross-references:** **LangGraph/11** (supervisor), **LangGraph/10** (subgraphs), **12. Memory and Conversation State**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json

try:
    import autogen_agentchat
    print("autogen-agentchat:", getattr(autogen_agentchat, "__version__", "installed"))
except ImportError:
    print("pip install autogen-agentchat autogen-ext[openai]")

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat, SelectorGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [ ]:
DOC_CHUNKS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE."},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions."},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header."},
    {"id": "c4", "text": "Pytest fixtures share database setup. Use conftest.py for session-scoped engines."},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files."},
]


def search_docs(query: str) -> str:
    """Keyword search over Notes API documentation chunks."""
    tokens = set(query.lower().split())
    hits = []
    for chunk in DOC_CHUNKS:
        if tokens & set(chunk["text"].lower().split()):
            hits.append(f"[{chunk['id']}] {chunk['text']}")
    return "\n".join(hits) if hits else "No matching chunks."


def describe_endpoint(method: str, path: str) -> str:
    """Describe a Notes API HTTP endpoint (teaching stub)."""
    catalog = {
        ("GET", "/notes"): "List all notes for the authenticated user.",
        ("POST", "/notes"): "Create a note; body requires title and content.",
        ("PUT", "/notes/{id}"): "Update an existing note by id.",
        ("DELETE", "/notes/{id}"): "Delete a note by id.",
    }
    return catalog.get((method.upper(), path), f"Unknown endpoint {method} {path}")

In [ ]:
def make_model_client():
    return OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

model_client = make_model_client()

In [ ]:
text_termination = TextMentionTermination("TERMINATE")
max_msg_termination = MaxMessageTermination(max_messages=20)
default_termination = text_termination | max_msg_termination

---

## 1. Pipeline vs Conversation

| Style | Mechanism | Order | Shared context |
|-------|-----------|-------|----------------|
| **Conversation** | `SelectorGroupChat` / `Swarm` | Dynamic | One thread, broadcast |
| **Pipeline** | Sequential `run` calls | Fixed A→B→C | Pass output as next task |
| **Hierarchical** | Manager delegates in chat | Manager-driven | One thread + role boundaries |

```text
Pipeline:   retrieve.run() → draft.run(task=retrieve_output)
Conversation: selector picks next speaker each turn
Hierarchical: manager assigns workers inside one team
```

---

## 2. Sequential Workflow — Agent A Then B

Pattern: run team **A** to termination, extract last message, start team **B** with that text as the new **task**:

In [ ]:
retriever = AssistantAgent(
    "retriever",
    description="Retrieves Notes API documentation chunks.",
    model_client=model_client,
    tools=[search_docs],
    system_message="Use search_docs. Return findings with [c#] citations only. End with DONE.",
)

drafter = AssistantAgent(
    "drafter",
    description="Drafts user-facing answers from retrieved context.",
    model_client=model_client,
    system_message="Write a concise answer using ONLY the provided context. End with TERMINATE.",
)

retrieve_team = RoundRobinGroupChat(
    [retriever],
    termination_condition=TextMentionTermination("DONE") | MaxMessageTermination(max_messages=6),
)
draft_team = RoundRobinGroupChat(
    [drafter],
    termination_condition=default_termination,
)

In [ ]:
async def sequential_notes_answer(question: str) -> str:
    r1 = await retrieve_team.run(task=question)
    context = r1.messages[-1].to_text() if hasattr(r1.messages[-1], "to_text") else str(r1.messages[-1].content)
    r2 = await draft_team.run(task=f"Context:\n{context}\n\nQuestion: {question}")
    return r2.messages[-1].to_text() if hasattr(r2.messages[-1], "to_text") else str(r2.messages[-1].content)


# answer = await sequential_notes_answer("How does JWT auth work?")
print("sequential: retrieve_team → draft_team")

**Note:** Call `await retrieve_team.reset()` and `await draft_team.reset()` between unrelated questions to avoid context bleed.

---

## 3. Sequential With Explicit `reset()` Between Stages

In [ ]:
async def sequential_with_reset(question: str):
    await retrieve_team.reset()
    await draft_team.reset()
    return await sequential_notes_answer(question)


print("Resets isolate each pipeline run")

---

## 4. Hierarchical Pattern — Manager Delegates to Workers

Manager agent **plans and assigns**; workers **execute** with tools. Use `SelectorGroupChat` with routing descriptions (**09**, **10**):

In [ ]:
manager = AssistantAgent(
    "manager",
    description="Notes API project manager. Delegates to workers. Speaks first and last.",
    model_client=model_client,
    system_message=(
        "You manage a Notes API team: doc_worker, api_worker.\n"
        "Assign one worker per subtask. Never use tools.\n"
        "Summarize results and end with TERMINATE."
    ),
)

doc_worker = AssistantAgent(
    "doc_worker",
    description="Worker: documentation search with search_docs.",
    model_client=model_client,
    tools=[search_docs],
    system_message="Execute assigned doc searches only.",
)

api_worker = AssistantAgent(
    "api_worker",
    description="Worker: endpoint descriptions with describe_endpoint.",
    model_client=model_client,
    tools=[describe_endpoint],
    system_message="Execute assigned endpoint lookups only.",
)

hierarchy_team = SelectorGroupChat(
    [manager, doc_worker, api_worker],
    model_client=model_client,
    termination_condition=default_termination,
)

---

## 5. Run Hierarchical Team

In [ ]:
HIER_TASK = "Prepare a onboarding blurb: Alembic migrations + POST /notes behavior."

# await Console(hierarchy_team.run_stream(task=HIER_TASK))
print("hierarchical SelectorGroupChat ready")

---

## 6. `MagenticOneGroupChat` Preview

**Magentic-One** is Microsoft's generalist multi-agent orchestrator (browser, coder, file surfer, etc.). In AgentChat:

```python
from autogen_agentchat.teams import MagenticOneGroupChat

# team = MagenticOneGroupChat([agent1, agent2, ...], model_client=model_client)
```

It provides a **lead orchestrator agent** inside the team — closer to a built-in **manager** than raw `SelectorGroupChat`. Use it for **open-ended** tasks; use custom hierarchies when you need **full control** over roles and tools.

| Team | Orchestration |
|------|---------------|
| `SelectorGroupChat` | External selector LLM |
| `MagenticOneGroupChat` | Bundled Magentic-One lead |
| Custom manager + workers | You write prompts + descriptions |

In [ ]:
try:
    from autogen_agentchat.teams import MagenticOneGroupChat
    M1_AVAILABLE = True
except ImportError:
    M1_AVAILABLE = False

print("MagenticOneGroupChat import:", M1_AVAILABLE)

---

## 7. Compare to LangGraph Supervisor (**LangGraph/11**)

| Aspect | AutoGen hierarchical team | LangGraph supervisor graph |
|--------|---------------------------|----------------------------|
| State | Shared chat thread | Typed `SupervisorState` + reducers |
| Routing | Selector LLM / descriptions | Conditional edges on `next_agent` |
| Workers | `AssistantAgent` + tools | Compiled ReAct subgraphs |
| Persistence | `save_state` / `load_state` (**12**) | Checkpointer + `thread_id` (**LangGraph/08**) |
| Visibility | `TaskResult.messages` | `stream_mode="updates"` |

Both patterns separate **routing** from **execution**; LangGraph makes graph structure explicit, AutoGen keeps chat-native ergonomics.

---

## 8. Pipeline Stage Diagram (Notes API)

```text
Question
   │
   ▼
┌─────────────┐
│  retriever  │  search_docs → "DONE"
└──────┬──────┘
       │ context string
       ▼
┌─────────────┐
│   drafter   │  prose answer → "TERMINATE"
└─────────────┘
```

Conversation team would let both agents speak in one loop; pipeline **forces** retrieval before drafting.

---

## 9. `build_notes_pipeline_team()` — Sequential Factory

In [ ]:
def build_notes_pipeline_team():
    retriever = AssistantAgent(
        "retriever",
        description="Retrieves documentation chunks.",
        model_client=model_client,
        tools=[search_docs],
        system_message="search_docs only; end with DONE.",
    )
    drafter = AssistantAgent(
        "drafter",
        description="Drafts answers from context.",
        model_client=model_client,
        system_message="Use provided context only; end with TERMINATE.",
    )
    stage1 = RoundRobinGroupChat(
        [retriever],
        termination_condition=TextMentionTermination("DONE") | MaxMessageTermination(max_messages=6),
    )
    stage2 = RoundRobinGroupChat(
        [drafter],
        termination_condition=default_termination,
    )
    return stage1, stage2


async def run_notes_pipeline(question: str) -> str:
    stage1, stage2 = build_notes_pipeline_team()
    await stage1.reset()
    await stage2.reset()
    r1 = await stage1.run(task=question)
    ctx = r1.messages[-1].to_text() if hasattr(r1.messages[-1], "to_text") else str(r1.messages[-1].content)
    r2 = await stage2.run(task=f"Context:\n{ctx}\n\nQuestion: {question}")
    return r2.messages[-1].to_text() if hasattr(r2.messages[-1], "to_text") else str(r2.messages[-1].content)


print("pipeline factory ready")

---

## 10. When to Choose Each Pattern

| Choose | When |
|--------|------|
| **Sequential pipeline** | Strict stage order, different models per stage, easy testing per stage |
| **SelectorGroupChat** | Dynamic expertise routing, collaborative debugging |
| **Swarm** | Agents already hand off explicitly (**10**) |
| **MagenticOneGroupChat** | Generalist automation, minimal custom orchestration |
| **LangGraph supervisor** | Need explicit graph, checkpointing, HITL (**LangGraph/09**) |

---

## 11. Compose Pipeline + Hierarchical

Advanced: pipeline **retrieval** then hierarchical **draft+review** team:

In [ ]:
async def pipeline_then_hierarchy(question: str):
    ctx_result = await retrieve_team.run(task=question)
    ctx = ctx_result.messages[-1].to_text() if hasattr(ctx_result.messages[-1], "to_text") else str(ctx_result.messages[-1].content)
    task = f"Using this context:\n{ctx}\n\nProduce final onboarding text for: {question}"
    return await hierarchy_team.run(task=task)


print("compose: sequential retrieve → hierarchical polish")

---

## 12. Termination Boundaries Between Stages

Use **distinct stop tokens** per stage (`DONE` vs `TERMINATE`) so stage 1 does not trigger global team stop early. See **13** for composing conditions with `|`.

In [ ]:
stage1_term = TextMentionTermination("DONE")
stage2_term = TextMentionTermination("TERMINATE")
print("Stage1 stops on DONE; stage2 on TERMINATE")

---

## 13. Async Streaming Sequential Stages

In [ ]:
async def stream_sequential(question: str):
    print("=== Stage 1: retrieve ===")
    await Console(retrieve_team.run_stream(task=question))
    r1 = await retrieve_team.run(task="")  # no-op read last state — prefer save last msg from stream
    print("=== Stage 2: draft ===")
    # In practice, capture last message from stage 1 result object instead.
    await retrieve_team.reset()
    await draft_team.reset()


print("Stream each stage for UX progress bars")

---

## 14. Testing Pipeline Stages Independently

Sequential design enables **unit-style** agent tests — run retriever alone with `MaxMessageTermination(3)` and assert `[c#]` appears in output (**06**).

In [ ]:
async def test_retriever_only():
    await retrieve_team.reset()
    result = await retrieve_team.run(task="Alembic upgrade head")
    text = " ".join(m.to_text() for m in result.messages if hasattr(m, "to_text"))
    return "c2" in text or "Alembic" in text


# ok = await test_retriever_only()
print("test retriever stage in isolation")

---

## 15. Summary

- **Sequential:** terminate stage A → pass output as task to stage B (`build_notes_pipeline_team`).
- **Hierarchical:** manager + workers in `SelectorGroupChat` (or `MagenticOneGroupChat` for bundled orchestration).
- **Pipeline vs conversation:** fixed stages vs dynamic multi-agent thread.
- **LangGraph/11** offers explicit supervisor graphs; AutoGen stays chat-first.

**Next:** **12. Memory and Conversation State** — `save_state`, contexts, and multi-turn loops.